# Heart Disease UCI Dataset — Exploratory Data Analysis

This notebook reuses the same cleaning logic as
`src/heart_disease/data.py` (installed as the `heart_disease` package) so the EDA here is
guaranteed to match what the training pipeline actually sees.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from heart_disease.data import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET,
    get_clean_dataset,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

## 1. Load & clean the dataset

`get_clean_dataset()` downloads the raw CSV from the UCI repository (if not
already present), parses `?` as missing, mode-imputes the handful of missing `ca`/`thal`
values, and binarizes the target (`num > 0 -> 1`).

In [ ]:
df = get_clean_dataset()
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 2. Missing value analysis

Should be all zeros — `ca` and `thal` had a small number of `?` values in the raw file,
resolved by mode imputation in `clean_data()`.

In [ ]:
missing = df.isna().sum()
print(missing)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=missing.index, y=missing.values, ax=ax)
ax.set_title("Missing values per column (post-cleaning)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 3. Class balance

In [ ]:
print(df[TARGET].value_counts(normalize=True))

fig, ax = plt.subplots(figsize=(5, 4))
counts = df[TARGET].value_counts().sort_index()
sns.barplot(x=counts.index.map({0: "No Disease", 1: "Disease"}), y=counts.values, ax=ax)
ax.set_title("Class Balance: Heart Disease Presence")
ax.set_ylabel("Count")
for i, v in enumerate(counts.values):
    ax.text(i, v + 3, str(v), ha="center")
plt.tight_layout()
plt.show()

## 4. Feature distributions (numeric features by class)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, NUMERIC_FEATURES):
    sns.histplot(data=df, x=col, hue=TARGET, kde=True, ax=ax, palette="Set1", alpha=0.6)
    ax.set_title(f"Distribution of {col}")
for ax in axes.flat[len(NUMERIC_FEATURES):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Correlation heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## 6. Feature relationships with the target

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x=TARGET, y="thalach", ax=ax)
ax.set_xticks([0, 1])
ax.set_xticklabels(["No Disease", "Disease"])
ax.set_title("Max Heart Rate Achieved vs. Disease Presence")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(data=df, x="cp", hue=TARGET, ax=ax)
ax.set_title("Chest Pain Type vs. Disease Presence")
plt.tight_layout()
plt.show()

## 7. Summary of findings

- Dataset is reasonably balanced (~54% no disease vs. ~46% disease presence) — no
  resampling required.
- No missing values remain after cleaning (`ca`/`thal` mode-imputed).
- `thal`, `ca`, `exang`, `oldpeak`, and `cp` show the strongest correlation with the target.
- `thalach` (max heart rate achieved) is negatively correlated with disease presence —
  patients without disease tend to reach a higher max heart rate during exercise testing.
- These signals carry through consistently to the feature importance / coefficients seen in
  the trained models in `notebooks/` training pipeline (`heart_disease.train`).